# Fine-Tuning Qwen2.5-0.5B-Instruct as a Resume Screener (LoRA)

Runs on Colab's **free T4 GPU**. Runtime: Runtime -> Change runtime type -> T4 GPU.

This fine-tunes a small open model to output structured JSON verdicts from resume text, using LoRA (Low-Rank Adaptation) so we only train a small number of extra parameters instead of the whole model.

## 1. Install dependencies

In [1]:
!pip install -q transformers==4.46.0 peft==0.13.2 trl==0.11.4 accelerate==1.0.1 datasets==3.0.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.4 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 10.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages 

## 2. Upload the dataset

Upload `train.jsonl` and `eval.jsonl` from the `data/` folder (generated locally by `generate_dataset.py`) using the file browser on the left, or run the cell below to upload directly.

In [3]:
from google.colab import files
uploaded = files.upload()  # select train.jsonl and eval.jsonl

Saving train.jsonl to train.jsonl


## 3. Load base model and tokenizer

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

## 4. Load and format the dataset

Each example becomes a single training string using the model's chat template: a user turn (the resume screening prompt) followed by an assistant turn (the JSON verdict).

In [5]:
from datasets import load_dataset

raw = load_dataset("json", data_files={"train": "train.jsonl", "eval": "eval.jsonl"})

def format_example(ex):
    messages = [
        {"role": "user", "content": ex["prompt"]},
        {"role": "assistant", "content": ex["completion"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_ds = raw["train"].map(format_example)
eval_ds = raw["eval"].map(format_example)

print(train_ds[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Screen this resume for a DevOps Engineer position and return a structured verdict.

Resume: Bachelor's degree in Computer Science or related field. 8 years working on cloud deployment at a mid-size company. Skilled in Ansible, Kubernetes, Docker, Linux, Jenkins, Terraform, CI/CD, AWS.<|im_end|>
<|im_start|>assistant
{"role": "DevOps Engineer", "ats_score": 100, "verdict": "strong_match", "matched_skills": ["Ansible", "Kubernetes", "Docker", "Linux", "Jenkins", "Terraform", "CI/CD", "AWS"], "missing_skills": [], "years_experience": 8}<|im_end|>



In [ ]:
!pip uninstall -y bitsandbytes

Found existing installation: bitsandbytes 0.44.1
Uninstalling bitsandbytes-0.44.1:
  Successfully uninstalled bitsandbytes-0.44.1


## 5. Configure LoRA

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect roughly 0.5-1% of total params trainable -- that's the point of LoRA

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


## 6. Train

In [8]:
import trl
print(trl.__version__)

0.11.4


In [9]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./resume-screener-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    per_device_eval_batch_size=4,
    max_seq_length=512,
    dataset_text_field="text",
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

trainer.train()

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


Epoch,Training Loss,Validation Loss
1,0.407800,0.278381
2,0.195600,0.188315
3,0.171300,0.171428


TrainOutput(global_step=150, training_loss=0.39782546122868856, metrics={'train_runtime': 464.8649, 'train_samples_per_second': 5.163, 'train_steps_per_second': 0.323, 'total_flos': 827324662348800.0, 'train_loss': 0.39782546122868856, 'epoch': 3.0})

## 7. Save the LoRA adapter

This saves only the small adapter weights (a few MB), not the full model -- that's what you download and use for inference/comparison.

In [ ]:
model.save_pretrained("./resume-screener-lora-final")
tokenizer.save_pretrained("./resume-screener-lora-final")

!zip -r resume-screener-lora-final.zip resume-screener-lora-final

from google.colab import files
files.download("resume-screener-lora-final.zip")

## 8. Quick sanity check before downloading

Run one example through the fine-tuned model right here in Colab to eyeball the output before you download anything.

In [10]:
test_prompt = (
    "Screen this resume for a Backend Engineer position and return a structured verdict.\n\n"
    "Resume: Bachelor's degree in Computer Science. 4 years of experience building distributed backend systems. "
    "Skilled in Python, PostgreSQL, Docker, Kubernetes."
)
messages = [{"role": "user", "content": test_prompt}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)

output = model.generate(inputs, max_new_tokens=150, do_sample=False)
print(tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos toke

{"role": "Backend Engineer", "ats_score": 56, "verdict": "moderate_match", "matched_skills": ["Python", "PostgreSQL", "Docker", "Kubernetes"], "missing_skills": ["Java", "REST APIs", "Redis"], "years_experience": 4}


In [11]:
trainer.save_model("./resume-screener-lora")
tokenizer.save_pretrained("./resume-screener-lora")

('./resume-screener-lora/tokenizer_config.json',
 './resume-screener-lora/special_tokens_map.json',
 './resume-screener-lora/vocab.json',
 './resume-screener-lora/merges.txt',
 './resume-screener-lora/added_tokens.json',
 './resume-screener-lora/tokenizer.json')

In [12]:
import shutil
shutil.make_archive("resume-screener-lora", 'zip', "./resume-screener-lora")

'/content/resume-screener-lora.zip'

In [13]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model.config._name_or_path if hasattr(model, "config") else "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

inputs2 = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(base_model.device)
output2 = base_model.generate(inputs2, max_new_tokens=150, do_sample=False)
print(tokenizer.decode(output2[0][inputs2.shape[1]:], skip_special_tokens=True))

Based on the provided resume, here is a structured verdict:

**Summary:** The candidate has a strong background in computer science with extensive experience in developing distributed backend systems using Python, PostgreSQL, Docker, and Kubernetes. They have demonstrated proficiency in these technologies through their work experience.

**Key Skills:**
- **Programming Languages:** Proficient in Python, PostgreSQL, and Docker.
- **Docker and Kubernetes:** Experience with Docker and Kubernetes.
- **Backend Systems Development:** Skilled in building scalable and resilient backend systems.
- **Project Management:** Strong project management skills to ensure timely delivery of projects.

**Experience:**
- **Years of Experience:** 4 years of relevant experience in building distributed backend systems.
- **Roles and Responsibilities:** Experienced as a
